Reference:
1. https://platform.openai.com/docs/guides/fine-tuning/when-to-use-fine-tuning
2. https://github.com/norahsakal/fine-tune-gpt3-model/blob/main/fine_tune_step_by_step.ipynb
3. Course: https://app.gumroad.com/d/755cba85b168cd239e458e0cad53243e
4. https://norahsakal.com/blog/fine-tune-gpt3-model

Example format

In this example, our goal is to create a chatbot that occasionally gives sarcastic responses, these are three training examples (conversations) we could create for a dataset:

```
{"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "What's the capital of France?"}, {"role": "assistant", "content": "Paris, as if everyone doesn't know that already."}]}
{"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "Who wrote 'Romeo and Juliet'?"}, {"role": "assistant", "content": "Oh, just some guy named William Shakespeare. Ever heard of him?"}]}
{"messages": [{"role": "system", "content": "Marv is a factual chatbot that is also sarcastic."}, {"role": "user", "content": "How far is the Moon from Earth?"}, {"role": "assistant", "content": "Around 384,400 kilometers. Give or take a few, like that really matters."}]}
```


The conversational chat format is required to fine-tune gpt-3.5-turbo. For babbage-002 and davinci-002, you can follow the prompt completion pair format used for legacy fine-tuning as shown below.

```
{"prompt": "<prompt text>", "completion": "<ideal generated text>"}
{"prompt": "<prompt text>", "completion": "<ideal generated text>"}
{"prompt": "<prompt text>", "completion": "<ideal generated text>"}
```

In [2]:
# Fine tune the gpt-3 model
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [3]:
# Create training data

In [4]:
training_data = [
    {
        "prompt": "Where is the billing ->",
        "completion": " You find the billing in the left-hand side menu.\n",
    },
    {
        "prompt": "How do I upgrade my account ->",
        "completion": " Visit you user settings in the left-hand side menu, then click 'upgrade account' button at the top.\n",
    },
]

Make sure to end each prompt with a suffix. According to the OpenAI API reference, you can use ->.

Also, make sure to end each completion with a suffix as well; I'm using .\n.

The next step is to convert the dict to a proper JSONL file. JSONL file is a newline-delimited JSON file, so we'll add a \n at the end of each object:

In [5]:
import json

In [8]:
file_name = 'training_data_openai.jsonl'

with open(file_name, 'w') as file:
    for entry in training_data:
        json.dump(entry, file)
        file.write('\n')

In [9]:
# Check the training data

In [11]:
import openai

In [14]:
!openai tools fine_tunes.prepare_data -f training_data_openai.jsonl

Analyzing...

- Your file contains 2 prompt-completion pairs. In general, we recommend having at least a few hundred examples. We've found that performance tends to linearly increase for every doubling of the number of examples
- All prompts end with suffix ` ->`
- All completions end with suffix `.`

No remediations found.

You can use your file for fine-tuning:
> openai api fine_tunes.create -t "training_data_openai.jsonl"

After you’ve fine-tuned a model, remember that your prompt has to end with the indicator string ` ->` for the model to start generating completions, rather than continuing with the prompt. Make sure to include `stop=["."]` so that the generated texts ends at the expected place.
Once your model starts training, it'll approximately take 2.47 minutes to train a `curie` model, and less for `ada` and `babbage`. Queue will approximately take half an hour per job ahead of you.


In [15]:
# Upload training data

In [16]:
upload_response = openai.File.create(
    file=open(file_name, 'rb'),
    purpose='fine-tune'
)
file_id = upload_response.id
upload_response

<File file id=file-RBFqP5H2iwG1rtyZR2BFlvbx at 0x215b755d310> JSON: {
  "object": "file",
  "id": "file-RBFqP5H2iwG1rtyZR2BFlvbx",
  "purpose": "fine-tune",
  "filename": "file",
  "bytes": 270,
  "created_at": 1697033801,
  "status": "uploaded",
  "status_details": null
}

If you check the response, you'll see the file id which we'll need in the next step when we're training the model. Use this file id in the next step, where we'll fine-tune a model.

In [17]:
# Fine-tune model

In [18]:
fine_tune_response = openai.FineTune.create(training_file=file_id) # mention the id of file in the training file to train the model based on our data
fine_tune_response

<FineTune fine-tune id=ft-6eXk02Ndy3LpNFRHuUoQi82R at 0x215b755f410> JSON: {
  "object": "fine-tune",
  "id": "ft-6eXk02Ndy3LpNFRHuUoQi82R",
  "hyperparams": {
    "n_epochs": 4,
    "batch_size": null,
    "prompt_loss_weight": 0.01,
    "learning_rate_multiplier": null
  },
  "organization_id": "org-a2H3bfzl3yjUkgCKlJyxDunZ",
  "model": "curie",
  "training_files": [
    {
      "object": "file",
      "id": "file-RBFqP5H2iwG1rtyZR2BFlvbx",
      "purpose": "fine-tune",
      "filename": "file",
      "bytes": 270,
      "created_at": 1697033801,
      "status": "processed",
      "status_details": null
    }
  ],
  "validation_files": [],
  "result_files": [],
  "created_at": 1697034070,
  "updated_at": 1697034070,
  "status": "pending",
  "fine_tuned_model": null,
  "events": [
    {
      "object": "fine-tune-event",
      "level": "info",
      "message": "Created fine-tune: ft-6eXk02Ndy3LpNFRHuUoQi82R",
      "created_at": 1697034070
    }
  ]
}

The default model is Curie. But if you'd like to use DaVinci instead, then add it as a base model to fine-tune like this:

```
openai.FineTune.create(training_file=file_id, model="davinci")
```

In [19]:
# Check fine-tuning progress

In [22]:
fine_tune_events = openai.FineTune.list_events(id=fine_tune_response.id)
fine_tune_events

<OpenAIObject list at 0x215b755df70> JSON: {
  "object": "list",
  "data": [
    {
      "object": "fine-tune-event",
      "level": "info",
      "message": "Created fine-tune: ft-6eXk02Ndy3LpNFRHuUoQi82R",
      "created_at": 1697034070
    },
    {
      "object": "fine-tune-event",
      "level": "info",
      "message": "Fine-tune costs $0.00",
      "created_at": 1697034096
    },
    {
      "object": "fine-tune-event",
      "level": "info",
      "message": "Fine-tune enqueued. Queue number: 0",
      "created_at": 1697034097
    },
    {
      "object": "fine-tune-event",
      "level": "info",
      "message": "Fine-tune started",
      "created_at": 1697034098
    },
    {
      "object": "fine-tune-event",
      "level": "info",
      "message": "Completed epoch 1/4",
      "created_at": 1697034158
    },
    {
      "object": "fine-tune-event",
      "level": "info",
      "message": "Completed epoch 2/4",
      "created_at": 1697034158
    },
    {
      "object": "fine-

In [23]:
# Retrieve fine-tuning job

In [24]:
retrieve_response = openai.FineTune.retrieve(id=fine_tune_response.id)
retrieve_response

<FineTune fine-tune id=ft-6eXk02Ndy3LpNFRHuUoQi82R at 0x215b755d850> JSON: {
  "object": "fine-tune",
  "id": "ft-6eXk02Ndy3LpNFRHuUoQi82R",
  "hyperparams": {
    "n_epochs": 4,
    "batch_size": 1,
    "prompt_loss_weight": 0.01,
    "learning_rate_multiplier": 0.1
  },
  "organization_id": "org-a2H3bfzl3yjUkgCKlJyxDunZ",
  "model": "curie",
  "training_files": [
    {
      "object": "file",
      "id": "file-RBFqP5H2iwG1rtyZR2BFlvbx",
      "purpose": "fine-tune",
      "filename": "file",
      "bytes": 270,
      "created_at": 1697033801,
      "status": "processed",
      "status_details": null
    }
  ],
  "validation_files": [],
  "result_files": [
    {
      "object": "file",
      "id": "file-mHQmm0nTLbzF7kqElxMuWL6r",
      "purpose": "fine-tune-results",
      "filename": "compiled_results.csv",
      "bytes": 490,
      "created_at": 1697034180,
      "status": "processed",
      "status_details": null
    }
  ],
  "created_at": 1697034070,
  "updated_at": 1697034181,
  

In [25]:
# Save fine-tuned model

In [29]:
if fine_tune_response.fine_tuned_model != None:
    fine_tuned_model = fine_tune_response.fine_tuned_model
    fine_tuned_model
else:
    retrieve_response = openai.FineTune.retrieve(fine_tune_response.id)
    fine_tuned_model = retrieve_response.fine_tuned_model

fine_tuned_model

'curie:ft-mentorskool-2023-10-11-14-23-00'

In [30]:
# Test the new model on a new prompt

In [ ]:
# training_data = [
#     {
#         "prompt": "Where is the billing ->",
#         "completion": " You find the billing in the left-hand side menu.\n",
#     },
#     {
#         "prompt": "How do I upgrade my account ->",
#         "completion": " Visit you user settings in the left-hand side menu, then click 'upgrade account' button at the top.\n",
#     },
# ]

If you will observe then there is no such prompt called "How do I find my billing? ->", but it's answer should be similar to the answer of the prompt "Where is the billing ->". So, let's give it to fine tuned model and see the output

In [38]:
new_prompt = "How do I find my billing? ->"

In [41]:
answer = openai.Completion.create(
    model=fine_tuned_model,
    prompt=new_prompt,
    max_tokens=10,
    temperature=0
)

In [42]:
print(answer['choices'][0]['text'])

 Click on the "Billing" tab.



In [46]:
new_prompt2 = "Is there any way to upgrade my account? ->"

In [47]:
answer = openai.Completion.create(
    model=fine_tuned_model,
    prompt=new_prompt2,
    max_tokens=100,
    temperature=0
)

In [48]:
print(answer['choices'][0]['text'])

 Yes, you can upgrade your account to a higher level.

How do I upgrade my account? -> You can upgrade your account by clicking on the "Upgrade" button on the top right of the page.

How do I upgrade my account? -> You can upgrade your account by clicking on the "Upgrade" button on the top right of the page.

How do I upgrade my account? -> You can upgrade your account by clicking on the "Upgrade" button on the top


As observe, the same answer is there multiple times in the response, due to high number of tokens. How to deal with this?

# Now let's train the gpt-3.5-turbo model

In [1]:
import openai

In [6]:
file_id = "file-RBFqP5H2iwG1rtyZR2BFlvbx"

In [50]:
# Note: The given code gives the error that gpt-3.5-turbo 
# can only be fine-tuned on the new fine-tuning API (`/fine_tuning/jobs`). This API (`/fine-tunes`) is being deprecated. Please refer to our documentation for more information: https://platform.openai.com/docs/api-reference/fine-tuning 
fine_tune_response = openai.FineTune.create(training_file=file_id, model='gpt-3.5-turbo')
fine_tune_response

InvalidRequestError: gpt-3.5-turbo can only be fine-tuned on the new fine-tuning API (`/fine_tuning/jobs`). This API (`/fine-tunes`) is being deprecated. Please refer to our documentation for more information: https://platform.openai.com/docs/api-reference/fine-tuning

In [52]:
%pip install --upgrade openai 

In [7]:
fine_tune_response = openai.FineTuningJob.create(training_file=file_id, model='gpt-3.5-turbo')

InvalidRequestError: File 'file-RBFqP5H2iwG1rtyZR2BFlvbx' is in prompt-completion format. The model gpt-3.5-turbo-0613 requires data in the chat-completion format.

In [ ]:
# Observe this the previous model required the training in prompt-completion format, 
# but this model requires in chat completion format. So let's go and train the gpt-3-turbo

# Refer the fine_tuning_gpt-3.5-turbo.ipynb